# 04 · 数据集与 DataLoader

> 数据是深度学习的燃料。这一节讲清楚 PyTorch 数据管道的三个组件：
> **Dataset（提供单条样本）→ Sampler（决定顺序）→ DataLoader（组 batch）**。

## 4.1 整体结构

```
┌─────────────┐    ┌──────────────┐    ┌──────────────┐
│  Dataset    │───▶│   Sampler    │───▶│  DataLoader  │───▶ batch
│ (取第 i 条) │    │ (洗牌/顺序)  │    │ (组批/多进程)│
└─────────────┘    └──────────────┘    └──────────────┘
```

- **Dataset**：实现 `__len__` 和 `__getitem__`。
- **DataLoader**：负责 `batch_size`、`shuffle`、多进程加载、`collate_fn`。

## 4.2 最简单的 Dataset：TensorDataset

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X = torch.randn(100, 5)              # 100 个样本，5 维特征
y = torch.randint(0, 3, (100,))      # 100 个标签，3 类

dataset = TensorDataset(X, y)
print("长度:", len(dataset))

# 取第 0 条样本
x0, y0 = dataset[0]
print("第 0 条: x", x0.shape, " y", y0)
print("取多条:", [t.shape for t in dataset[0:4]])

## 4.3 DataLoader 基本用法

```python
DataLoader(dataset, batch_size=32, shuffle=True, num_workers=0, drop_last=False)
```

| 参数 | 含义 |
|---|---|
| `batch_size` | 每个批次样本数 |
| `shuffle` | 是否打乱（训练集 True，验证/测试集 False） |
| `num_workers` | 子进程数；Windows 建议 0 |
| `drop_last` | 丢弃最后不足一个 batch 的数据 |
| `pin_memory` | 锁页内存，GPU 训练时设 True 加速 |
| `collate_fn` | 自定义如何把样本组成 batch |

In [ ]:
loader = DataLoader(dataset, batch_size=16, shuffle=True, drop_last=False)

print("batch 数量:", len(loader))
print("总样本:", len(loader.dataset))

for i, (xb, yb) in enumerate(loader):
    print(f"batch {i}: x {tuple(xb.shape)}, y {tuple(yb.shape)}")
    if i == 2:
        break

In [ ]:
# drop_last 对比
print("100/16 向上取整:", len(DataLoader(dataset, batch_size=16)))
print("100/16 drop_last:", len(DataLoader(dataset, batch_size=16, drop_last=True)))
# 100 / 16 = 6.25 -> 不丢弃 7 个 batch，丢弃则为 6 个

# shuffle 的影响
a = next(iter(DataLoader(dataset, batch_size=5, shuffle=False)))[1]
b = next(iter(DataLoader(dataset, batch_size=5, shuffle=False)))[1]
print("\nshuffle=False 两次结果相同:", torch.equal(a, b))

> **训练集 shuffle=True，测试集 shuffle=False**（保证可复现，顺序无关）。

## 4.4 自定义 Dataset

最通用的做法：继承 `torch.utils.data.Dataset`。

**必须实现**：
- `__init__`：加载数据路径或数据
- `__len__`：返回样本数
- `__getitem__(index)`：返回第 index 条样本（通常返回 (x, y)）

In [ ]:
import numpy as np
from torch.utils.data import Dataset

class MyDataset(Dataset):
    def __init__(self, n=200, dim=10, seed=0):
        rng = np.random.RandomState(seed)
        self.x = torch.tensor(rng.randn(n, dim), dtype=torch.float32)
        # 造一个规律：标签由前两个特征之和决定（3 分类）
        s = self.x[:, 0] + self.x[:, 1]
        self.y = torch.where(s > 0.5, 2, torch.where(s > -0.5, 1, 0)).long()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

ds = MyDataset()
print("len:", len(ds))
x, y = ds[0]
print("样本 0:", x.shape, y.item())

class_counts = torch.bincount(ds.y)
print("各类样本数:", class_counts.tolist())

### 读取真实文件的自定义 Dataset（图像示例）

思路：`__init__` 里收集所有文件路径与标签；`__getitem__` 里按需读取并预处理。
关键点是**不要在 `__init__` 里把所有图片读进内存**（除非数据很小）。

In [ ]:
import os
from PIL import Image
from torchvision import transforms

class ImageFolderDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        self.samples = []      # [(path, label), ...]
        self.classes = sorted([
            d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))
        ])
        for label, cls in enumerate(self.classes):
            cls_dir = os.path.join(root, cls)
            for fname in os.listdir(cls_dir):
                if fname.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                    self.samples.append((os.path.join(cls_dir, fname), label))
        if len(self.samples) == 0:
            raise RuntimeError(f"在 {root} 下没找到图片，请检查路径结构")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

# 用法（需要有数据时）：
# ds = ImageFolderDataset("data/train", transform=transforms.ToTensor())

## 4.5 transform：数据预处理与数据增强

`torchvision.transforms` 提供常用图像变换。

- **训练**：随机裁剪、翻转、颜色抖动（数据增强，防过拟合）
- **验证/测试**：只做 resize + 标准化

**标准化公式**：`output = (input - mean) / std`，mean/std 通常是 ImageNet 统计值。

In [ ]:
from torchvision import transforms

train_tf = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomHorizontalFlip(p=0.5),        # 随机水平翻转
    transforms.RandomCrop(32, padding=4),          # 随机裁剪
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),                         # PIL/np -> [0,1] 张量 (C,H,W)
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

test_tf = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

from PIL import Image
img = Image.fromarray((np.random.rand(40, 40, 3) * 255).astype("uint8"))
t = train_tf(img)
print("transform 输出:", t.shape, t.dtype)
print("像素范围(标准化后):", t.min().item(), t.max().item())

### 常用的 transforms

| Transform | 作用 |
|---|---|
| `Resize` | 缩放 |
| `CenterCrop` / `RandomCrop` | 中心/随机裁剪 |
| `RandomHorizontalFlip` | 随机水平翻转 |
| `RandomRotation` | 随机旋转 |
| `ColorJitter` | 亮度/对比度/饱和度抖动 |
| `ToTensor` | 转成 (C,H,W) float 张量并归一化到 [0,1] |
| `Normalize` | 标准化 |
| `RandomErasing` | 随机遮挡（模拟遮挡） |
| `Lambda` | 自定义函数 |

也可以对张量做增强：`transforms.RandomErasing()` 需在 ToTensor 之后。

## 4.6 内置与下载数据集

`torchvision.datasets` 内置了大量常用数据集，会自动下载。

```python
from torchvision import datasets
datasets.MNIST(root="./data", train=True, download=True, transform=...)
```

In [ ]:
from torchvision import datasets, transforms

tf = transforms.Compose([transforms.ToTensor()])

# 会自动下载到 ./data（第一次运行需要联网，约 10MB）
train_set = datasets.MNIST(root="./data", train=True, download=True, transform=tf)
test_set = datasets.MNIST(root="./data", train=False, download=True, transform=tf)

print("训练集:", len(train_set), " 测试集:", len(test_set))
img, label = train_set[0]
print("单张图像:", img.shape, img.dtype, " 标签:", label)
print("像素范围:", img.min().item(), img.max().item())

In [ ]:
# 可视化几张（若已成功下载）
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 6, figsize=(12, 2))
for ax, i in zip(axes, range(6)):
    img, label = train_set[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(f"label={label}")
    ax.axis("off")
plt.tight_layout()
plt.show()

### 其它常用内置数据集

| 数据集 | 调用 |
|---|---|
| MNIST | `datasets.MNIST` |
| FashionMNIST | `datasets.FashionMNIST` |
| CIFAR10/100 | `datasets.CIFAR10` |
| ImageNet | `datasets.ImageFolder`（本地） |
| COCO 检测 | `datasets.CocoDetection` |

自定义文件夹结构（ImageFolder 要求）：
```
data/train/
├── cat/  a.jpg b.jpg ...
├── dog/  c.jpg d.jpg ...
└── bird/ e.jpg ...
```
对应 `datasets.ImageFolder("data/train", transform=tf)`。

## 4.7 划分训练/验证集

方法一：先划分再建 Dataset（用 `random_split`）。

In [ ]:
from torch.utils.data import random_split

full = datasets.MNIST(root="./data", train=True, download=True, transform=tf)
n_train = int(len(full) * 0.9)
n_val = len(full) - n_train

train_set, val_set = random_split(full, [n_train, n_val],
                                  generator=torch.Generator().manual_seed(42))
print("train:", len(train_set), " val:", len(val_set))

In [ ]:
# 方法二：Subset（用索引）
from torch.utils.data import Subset

idx = torch.randperm(len(full))
val_idx = idx[:5000]
tr_idx = idx[5000:]
train_sub = Subset(full, tr_idx)
val_sub = Subset(full, val_idx)
print("Subset train:", len(train_sub), " val:", len(val_sub))

## 4.8 collate_fn：自定义组批

默认 `collate_fn` 会对每个字段做 `torch.stack`。当样本长度不一（如变长文本）时需要自定义。

例如按 batch 内最长补齐（padding）：

In [ ]:
from torch.nn.utils.rnn import pad_sequence

class VariableLenDataset(Dataset):
    def __init__(self, n=10):
        self.data = [torch.randn(torch.randint(3, 8, (1,)).item(), 4) for _ in range(n)]
    def __len__(self):
        return len(self.data)
    def __getitem__(self, i):
        return self.data[i], torch.tensor(i % 2)

def collate_pad(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    padded = pad_sequence(seqs, batch_first=True)      # (B, T_max, 4)
    return padded, lengths, torch.stack(labels)

loader = DataLoader(VariableLenDataset(), batch_size=4, collate_fn=collate_pad)
for padded, lengths, labels in loader:
    print("padded:", tuple(padded.shape), " lengths:", lengths.tolist(), " labels:", labels.tolist())
    break

## 4.9 性能：num_workers 与 pin_memory

- `num_workers>0`：用多个子进程并行读数据，CPU 密集的预处理会更快。
- `pin_memory=True`：把数据放锁页内存，GPU 拷贝更快。
- **Windows / Jupyter 下 `num_workers>0` 常见报错**，建议先设 0，
  或在 `if __name__ == "__main__":` 保护下运行。

```python
train_loader = DataLoader(
    train_set, batch_size=64, shuffle=True,
    num_workers=4, pin_memory=True, persistent_workers=True,
)
```

In [ ]:
# 实测 DataLoader 吞吐（简单对比）
import time

big = TensorDataset(torch.randn(5000, 100), torch.randint(0, 10, (5000,)))

t0 = time.time()
for _ in DataLoader(big, batch_size=64, shuffle=True):
    pass
print(f"num_workers=0: {time.time()-t0:.3f}s")

t0 = time.time()
for _ in DataLoader(big, batch_size=64, shuffle=True, num_workers=0):
    pass
print(f"num_workers=0 (again): {time.time()-t0:.3f}s")

## 4.10 练习

**练习 1**：写一个自定义 Dataset，生成 500 个二维点，
按点是否在单位圆内标注 0/1。

**练习 2**：用 DataLoader 以 batch_size=32 遍历，统计正负样本比例。

**练习 3**：为上面的数据集加上"随机加高斯噪声"的增强（用 transforms.Lambda 思路），
并比较加噪前后模型的损失（可选）。

**练习 4**：用 `random_split` 把 MNIST 拆成 80% / 20%，并各自建 DataLoader。

In [ ]:
# 练习 1
class CircleDataset(Dataset):
    def __init__(self, n=500, seed=1):
        g = torch.Generator().manual_seed(seed)
        self.x = torch.rand(n, 2, generator=g) * 2 - 1     # [-1,1]
        self.y = (self.x.norm(dim=1) < 0.6).long()          # 圆内=1
    def __len__(self):
        return len(self.x)
    def __getitem__(self, i):
        return self.x[i], self.y[i]

ds = CircleDataset()
print("练习1 正负样本:", torch.bincount(ds.y).tolist())

# 练习 2
loader = DataLoader(ds, batch_size=32, shuffle=True)
pos = tot = 0
for xb, yb in loader:
    pos += yb.sum().item(); tot += len(yb)
print(f"练习2 正样本比例: {pos/tot:.3f}")

# 练习 3
noisy = CircleDataset()
noisy.x = noisy.x + 0.2 * torch.randn_like(noisy.x)
print("练习3 加噪后 x 标准差:", noisy.x.std().item(), " 原始:", ds.x.std().item())

# 练习 4
mnist = datasets.MNIST(root="./data", train=True, download=True, transform=tf)
n1 = int(0.8 * len(mnist))
tr, va = random_split(mnist, [n1, len(mnist) - n1],
                      generator=torch.Generator().manual_seed(0))
print("\n练习4:", len(tr), len(va))

## 4.11 小结

- 三件套：`Dataset`（单条）→ `Sampler`（顺序）→ `DataLoader`（成批）。
- 自定义 Dataset 实现 `__init__` / `__len__` / `__getitem__`。
- 训练集 `shuffle=True`；`num_workers` 加速（Windows 慎用）。
- `transforms` 做预处理与数据增强，训练用强增强、测试只用 resize+normalize。
- 变长序列用自定义 `collate_fn` + `pad_sequence`。

下一节：`05_训练流程与优化器.ipynb`